# 03. PD Logistic Regression

The target models probability of good standing (`good_bad = 1`). Default probability is calculated in the next notebook as `1 - P(good)`.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import norm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

DATA_PATH = Path('../data/loan_data_2007_2014.csv')
data = pd.read_csv(DATA_PATH, low_memory=False)
bad_statuses = {'Charged Off', 'Default', 'Does not meet the credit policy. Status:Charged Off', 'Late (31-120 days)'}
data['good_bad'] = (~data['loan_status'].isin(bad_statuses)).astype(int)

# Exact handoff from notebook 02. The broader model adds established candidate variables for comparison.
refined_raw_features = ['grade', 'term', 'verification_status', 'int_rate', 'dti']
broad_raw_features = refined_raw_features + [
    'annual_inc', 'emp_length', 'home_ownership', 'purpose', 'addr_state', 'initial_list_status'
]
broad_X = data[broad_raw_features].copy()
y = data['good_bad'].copy()
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    broad_X, y, test_size=0.2, random_state=42, stratify=y
)

# Fit numeric imputation on training rows only, then apply it unchanged to validation rows.
numeric_features = ['int_rate', 'dti', 'annual_inc']
training_medians = X_train_raw[numeric_features].median()
X_train_raw[numeric_features] = X_train_raw[numeric_features].fillna(training_medians)
X_test_raw[numeric_features] = X_test_raw[numeric_features].fillna(training_medians)

categorical_features = ['grade', 'term', 'verification_status', 'emp_length', 'home_ownership', 'purpose', 'addr_state', 'initial_list_status']
X_train = pd.get_dummies(X_train_raw, columns=categorical_features, dtype=int)
X_test = pd.get_dummies(X_test_raw, columns=categorical_features, dtype=int)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)
X_train = X_train.drop(columns=['grade_G'], errors='ignore')
X_test = X_test.drop(columns=['grade_G'], errors='ignore')
X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

## Broader and refined logistic models

The broader model includes the exact refined handoff plus income, employment, housing, purpose, state, and listing-status candidates. The refined model keeps only `grade`, `term`, `verification_status`, `int_rate`, and `dti`; both estimate probability of good standing.

In [2]:
broad_features = X_train.columns.tolist()
broad_model = LogisticRegression(max_iter=1000, solver='liblinear')
broad_model.fit(X_train[broad_features], y_train)
broad_auc = roc_auc_score(y_test, broad_model.predict_proba(X_test[broad_features])[:, 1])

refined_features = [
    column for column in X_train.columns
    if column.startswith(('grade_', 'term_', 'verification_status_')) or column in ['int_rate', 'dti']
]
refined_model = LogisticRegression(max_iter=1000, solver='liblinear')
refined_model.fit(X_train[refined_features], y_train)
refined_prob_good = refined_model.predict_proba(X_test[refined_features])[:, 1]
refined_auc = roc_auc_score(y_test, refined_prob_good)
pd.DataFrame({'model': ['broader', 'refined'], 'AUC': [broad_auc, refined_auc], 'Gini': [2 * broad_auc - 1, 2 * refined_auc - 1]})

,model,AUC,Gini
0,broader,0.556554,0.113109
1,refined,0.657632,0.315264


In [3]:
coefficients = pd.DataFrame({'feature': refined_features, 'coefficient': refined_model.coef_[0]})
coefficients['odds_ratio'] = np.exp(coefficients['coefficient'])
coefficients.sort_values('coefficient').head(20)

,feature,coefficient,odds_ratio
0,int_rate,-0.112465,0.893629
1,dti,-0.009099,0.990942
5,grade_D,-0.007058,0.992967
4,grade_C,0.016298,1.016432
6,grade_E,0.022462,1.022716
7,grade_F,0.083980,1.087607
3,grade_B,0.091860,1.096211
2,grade_A,0.381644,1.464690
12,verification_status_Verified,0.651742,1.918881
10,verification_status_Not Verified,0.693699,2.001103


In [4]:
# Approximate Wald p-values are shown for educational interpretation, not formal model approval.
design = np.c_[np.ones(len(X_train)), X_train[refined_features].to_numpy()]
probability = refined_model.predict_proba(X_train[refined_features])[:, 1]
covariance = np.linalg.pinv(design.T @ (design * (probability * (1 - probability))[:, None]))
standard_error = np.sqrt(np.diag(covariance))[1:]
coefficients['p_value'] = 2 * norm.sf(np.abs(coefficients['coefficient'] / standard_error))
coefficients.sort_values('p_value').head(20)

,feature,coefficient,odds_ratio,p_value
9,term_ 60 months,1.107448,3.026623,2.422402e-278
11,verification_status_Source Verified,0.766127,2.151418,3.788382e-278
8,term_ 36 months,1.004121,2.729506,1.043429e-228
10,verification_status_Not Verified,0.693699,2.001103,3.227511e-228
12,verification_status_Verified,0.651742,1.918881,4.522480e-188
0,int_rate,-0.112465,0.893629,2.412598e-165
1,dti,-0.009099,0.990942,6.082265e-41
2,grade_A,0.381644,1.464690,9.374819e-06
7,grade_F,0.083980,1.087607,8.909005e-02
3,grade_B,0.091860,1.096211,1.953833e-01


## Next stage

Notebook 04 converts `P(good)` to PD, reports AUC/Gini and a confusion matrix, then maps the refined model to an illustrative 300–850 scorecard.